# **Procurement & Vendor Analytics**




### **Data Exploration and Quality Assessment**
####**The objective of this Section:**

* Understand the structure of all four datasets.
Assess data quality.
* Identify missing values and duplicates.
* Explore relationships between datasets.
* Determine potential keys for joining tables.

##Imported Liabraries

In [ ]:
!pip install pandas numpy matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/Procurement")
print(list(DATA_PATH.iterdir()))

[PosixPath('/content/drive/MyDrive/Procurement/Procurement_Vendor_Analytics.ipynb'), PosixPath('/content/drive/MyDrive/Procurement/purchase_prices.csv'), PosixPath('/content/drive/MyDrive/Procurement/vendor_invoice.csv'), PosixPath('/content/drive/MyDrive/Procurement/vendor_sales_summary.csv'), PosixPath('/content/drive/MyDrive/Procurement/purchases.csv'), PosixPath('/content/drive/MyDrive/Procurement/.ipynb_checkpoints')]


In [ ]:
purchase_prices = pd.read_csv(DATA_PATH / "purchase_prices.csv")
purchases = pd.read_csv(DATA_PATH / "purchases.csv")
vendor_invoice = pd.read_csv(DATA_PATH / "vendor_invoice.csv")
vendor_sales_summary = pd.read_csv(DATA_PATH / "vendor_sales_summary.csv")

In [ ]:
print('Purchase Prices :', purchase_prices.shape)
purchase_prices.head()

Purchase Prices : (12261, 9)


,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,58,Gekkeikan Black & Gold Sake,12.99,750mL,750,1,9.28,8320,SHAW ROSS INT L IMP LTD
1,62,Herradura Silver Tequila,36.99,750mL,750,1,28.67,1128,BROWN-FORMAN CORP
2,63,Herradura Reposado Tequila,38.99,750mL,750,1,30.46,1128,BROWN-FORMAN CORP
3,72,No. 3 London Dry Gin,34.99,750mL,750,1,26.11,9165,ULTRA BEVERAGE COMPANY LLP
4,75,Three Olives Tomato Vodka,14.99,750mL,750,1,10.94,7245,PROXIMO SPIRITS INC.


In [ ]:
print('Purchases',purchases.shape)
purchases.head()

Purchases (2372474, 16)


,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1


In [ ]:
print('Vendor Invoice',vendor_invoice.shape)
vendor_invoice.head()

Vendor Invoice (5543, 10)


,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,NaN
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,NaN


In [ ]:
print('Vendor Sales',vendor_sales_summary.shape)
vendor_sales_summary.head()

Vendor Sales (10692, 18)


,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesPrice,TotalSalesDollars,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalestoPurchaseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750.0,145080,3811251.60,142049.0,672819.31,5101919.51,260999.20,68601.68,1290667.91,25.297693,0.979108,1.338647
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750.0,164038,3804041.22,160247.0,561512.37,4819073.49,294438.66,144929.24,1015032.27,21.062810,0.976890,1.266830
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750.0,187407,3418303.68,187140.0,461140.15,4538120.60,343854.07,123780.22,1119816.92,24.675786,0.998575,1.327594
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750.0,201682,3261197.94,200412.0,420050.01,4475972.88,368242.80,257032.07,1214774.94,27.139908,0.993703,1.372493
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750.0,138109,3023206.01,135838.0,545778.28,4223107.62,249587.83,257032.07,1199901.61,28.412764,0.983556,1.396897


In [ ]:
purchase_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12261 entries, 0 to 12260
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           12261 non-null  int64  
 1   Description     12260 non-null  object 
 2   Price           12261 non-null  float64
 3   Size            12260 non-null  object 
 4   Volume          12260 non-null  object 
 5   Classification  12261 non-null  int64  
 6   PurchasePrice   12261 non-null  float64
 7   VendorNumber    12261 non-null  int64  
 8   VendorName      12261 non-null  object 
dtypes: float64(2), int64(3), object(4)
memory usage: 862.2+ KB


In [ ]:
purchases.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2372474 entries, 0 to 2372473
Data columns (total 16 columns):
 #   Column          Dtype  
---  ------          -----  
 0   InventoryId     object 
 1   Store           int64  
 2   Brand           int64  
 3   Description     object 
 4   Size            object 
 5   VendorNumber    int64  
 6   VendorName      object 
 7   PONumber        int64  
 8   PODate          object 
 9   ReceivingDate   object 
 10  InvoiceDate     object 
 11  PayDate         object 
 12  PurchasePrice   float64
 13  Quantity        int64  
 14  Dollars         float64
 15  Classification  int64  
dtypes: float64(2), int64(6), object(8)
memory usage: 289.6+ MB


In [ ]:
vendor_invoice.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5543 entries, 0 to 5542
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   VendorNumber  5543 non-null   int64  
 1   VendorName    5543 non-null   object 
 2   InvoiceDate   5543 non-null   object 
 3   PONumber      5543 non-null   int64  
 4   PODate        5543 non-null   object 
 5   PayDate       5543 non-null   object 
 6   Quantity      5543 non-null   int64  
 7   Dollars       5543 non-null   float64
 8   Freight       5543 non-null   float64
 9   Approval      374 non-null    object 
dtypes: float64(2), int64(3), object(5)
memory usage: 433.2+ KB


In [ ]:
vendor_sales_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10692 entries, 0 to 10691
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorNumber           10692 non-null  int64  
 1   VendorName             10692 non-null  object 
 2   Brand                  10692 non-null  int64  
 3   Description            10692 non-null  object 
 4   PurchasePrice          10692 non-null  float64
 5   ActualPrice            10692 non-null  float64
 6   Volume                 10692 non-null  float64
 7   TotalPurchaseQuantity  10692 non-null  int64  
 8   TotalPurchaseDollars   10692 non-null  float64
 9   TotalSalesQuantity     10692 non-null  float64
 10  TotalSalesPrice        10692 non-null  float64
 11  TotalSalesDollars      10692 non-null  float64
 12  TotalExciseTax         10692 non-null  float64
 13  FreightCost            10692 non-null  float64
 14  GrossProfit            10692 non-null  float64
 15  Pr

### Missing Value Analysis

In [ ]:
files_dict = {
    "Purchases": purchases,
    "Purchase Prices": purchase_prices,
    "Vendor Invoice": vendor_invoice,
    "Vendor Sales Summary": vendor_sales_summary
}
for name, df in files_dict.items():
    print(f"\n{name}")

    missing_df = pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing %': round((df.isnull().sum() / len(df)) * 100, 2)
    })

    missing_df = missing_df[missing_df['Missing Count'] > 0]

    if not missing_df.empty:
        display(missing_df.sort_values('Missing %', ascending=False))
    else:
        print("No missing values found.")



Purchases


,Missing Count,Missing %
Size,3,0.0



Purchase Prices


,Missing Count,Missing %
Description,1,0.01
Size,1,0.01
Volume,1,0.01



Vendor Invoice


,Missing Count,Missing %
Approval,5169,93.25



Vendor Sales Summary
No missing values found.


In [ ]:
for name, df in files_dict.items():
    print(f"\n{name}")
    print(f"Duplicate Rows: {df.duplicated().sum()}")


Purchases
Duplicate Rows: 0

Purchase Prices
Duplicate Rows: 0

Vendor Invoice
Duplicate Rows: 0

Vendor Sales Summary
Duplicate Rows: 0


Basic Statistics

In [ ]:
for name, df in files_dict.items():
    print(f"\n{'='*60}")
    print(name)
    display(df.describe())


Purchases


,Store,Brand,VendorNumber,PONumber,PurchasePrice,Quantity,Dollars,Classification
count,2.372474e+06,2.372474e+06,2.372474e+06,2.372474e+06,2.372474e+06,2.372474e+06,2.372474e+06,2.372474e+06
mean,4.465133e+01,1.241864e+04,6.886436e+03,1.104094e+04,1.205005e+01,1.415585e+01,1.356815e+02,1.443520e+00
std,2.351245e+01,1.255728e+04,8.066694e+03,1.565340e+03,1.794510e+01,2.344616e+01,2.816649e+02,4.967999e-01
min,1.000000e+00,5.800000e+01,2.000000e+00,8.106000e+03,0.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00
25%,2.500000e+01,3.639000e+03,3.252000e+03,9.761000e+03,6.120000e+00,6.000000e+00,4.926000e+01,1.000000e+00
50%,4.800000e+01,6.523000e+03,4.425000e+03,1.110300e+04,9.220000e+00,1.000000e+01,8.393000e+01,1.000000e+00
75%,6.700000e+01,1.887700e+04,9.552000e+03,1.239700e+04,1.449000e+01,1.200000e+01,1.405200e+02,2.000000e+00
max,8.100000e+01,9.063100e+04,2.013590e+05,1.366100e+04,5.681810e+03,3.816000e+03,5.017570e+04,2.000000e+00



Purchase Prices


,Brand,Price,Classification,PurchasePrice,VendorNumber
count,12261.000000,12261.000000,12261.000000,12261.000000,12261.000000
mean,17989.067123,38.640240,1.708996,26.488220,10814.861757
std,12528.503464,206.151172,0.454244,156.182948,19007.682322
min,58.000000,0.000000,1.000000,0.000000,2.000000
25%,5990.000000,10.990000,1.000000,6.890000,3960.000000
50%,18788.000000,15.990000,2.000000,10.650000,7153.000000
75%,25117.000000,29.990000,2.000000,20.130000,9552.000000
max,90631.000000,13999.900000,2.000000,11111.030000,173357.000000



Vendor Invoice


,VendorNumber,PONumber,Quantity,Dollars,Freight
count,5543.00000,5543.000000,5543.000000,5.543000e+03,5543.000000
mean,20662.75212,10889.419087,6058.880931,5.807338e+04,295.954301
std,34582.15841,1600.859969,14453.338164,1.402340e+05,713.585093
min,2.00000,8106.000000,1.000000,4.140000e+00,0.020000
25%,3089.00000,9503.500000,83.000000,9.678100e+02,5.020000
50%,7240.00000,10890.000000,423.000000,4.765450e+03,24.730000
75%,10754.00000,12275.500000,5100.500000,4.458718e+04,229.660000
max,201359.00000,13661.000000,141660.000000,1.660436e+06,8468.220000



Vendor Sales Summary


,VendorNumber,Brand,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesPrice,TotalSalesDollars,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalestoPurchaseRatio
count,10692.000000,10692.000000,10692.000000,10692.000000,10692.000000,10692.000000,1.069200e+04,10692.000000,10692.000000,1.069200e+04,10692.000000,10692.000000,1.069200e+04,1.069200e+04,10692.000000,10692.000000
mean,10650.649458,18039.228769,24.385303,35.643671,847.360550,3140.886831,3.010669e+04,3077.482136,18793.783627,4.223907e+04,1774.226259,61433.763214,1.213238e+04,-inf,1.706793,2.504390
std,18753.519148,12662.187074,109.269375,148.246016,664.309212,11095.086769,1.230678e+05,10952.851391,44952.773386,1.676553e+05,10975.582240,60938.458032,4.622434e+04,NaN,6.020460,8.459067
min,2.000000,58.000000,0.360000,0.490000,50.000000,1.000000,7.100000e-01,0.000000,0.000000,0.000000e+00,0.000000,0.090000,-5.200278e+04,-inf,0.000000,0.000000
25%,3951.000000,5793.500000,6.840000,10.990000,750.000000,36.000000,4.534575e+02,33.000000,289.710000,7.292200e+02,4.800000,14069.870000,5.292000e+01,1.332451e+01,0.807229,1.153729
50%,7153.000000,18761.500000,10.455000,15.990000,750.000000,262.000000,3.655465e+03,261.000000,2857.800000,5.298045e+03,46.570000,50293.620000,1.399640e+03,3.040546e+01,0.981529,1.436894
75%,9552.000000,25514.250000,19.482500,28.990000,750.000000,1975.750000,2.073824e+04,1929.250000,16059.562500,2.839692e+04,418.650000,79528.990000,8.660200e+03,3.995613e+01,1.039342,1.665449
max,201359.000000,90631.000000,5681.810000,7499.990000,20000.000000,337660.000000,3.811252e+06,334939.000000,672819.310000,5.101920e+06,368242.800000,257032.070000,1.290668e+06,9.971666e+01,274.500000,352.928571


### Categorical Summary

In [ ]:
for name, df in files_dict.items():
    print(f"\n{'='*60}")
    print(name)

    display(df.describe(include='object'))


Purchases


,InventoryId,Description,Size,VendorName,PODate,ReceivingDate,InvoiceDate,PayDate
count,2372474,2372474,2372471,2372474,2372474,2372474,2372474,2372474
unique,245907,9652,51,129,319,364,373,382
top,73_DONCASTER_8068,Tito's Handmade Vodka,750mL,DIAGEO NORTH AMERICA INC,2024-06-27,2024-07-05,2024-08-16,2024-09-10
freq,180,16442,1207700,243326,24752,19640,29989,19713



Purchase Prices


,Description,Size,Volume,VendorName
count,12260,12260,12260,12261
unique,11114,56,33,136
top,Bacardi Superior Rum,750mL,750,MARTIGNETTI COMPANIES
freq,7,9823,9840,1631



Vendor Invoice


,VendorName,InvoiceDate,PODate,PayDate,Approval
count,5543,5543,5543,5543,374
unique,129,373,319,382,1
top,AMERICAN VINTAGE BEVERAGE,2024-07-04,2024-06-25,2024-08-14,Frank Delahunt
freq,55,43,86,31,374



Vendor Sales Summary


,VendorName,Description
count,10692,10692
unique,128,9651
top,MARTIGNETTI COMPANIES,Southern Comfort
freq,1388,10


## **Key Findings**

1. Four datasets were successfully loaded and explored:
* `purchases` (2,372,474 records, 16 columns)  
* `purchase_prices` (12,261 records, 9 columns)  
* `vendor_invoice` (5,543 records, 10 columns)  
* `vendor_sales_summary` (10,692 records, 18 columns)  

2. The Purchases dataset has the largest volume of data, making it a key source for trend and performance analysis.  

3. Missing values were identified:  
* `purchase_prices`: Description (1 missing, ~0.01%), Size (1 missing, ~0.01%), Volume (1 missing, ~0.01%)  
* `vendor_invoice`: Approval (5,169 missing, ~93.3%)  
Recommendation: Impute or validate missing values using reference data or vendor confirmations to ensure completeness before analysis.  

4. No duplicate rows were detected across any dataset, confirming data integrity.  

5. Date columns (e.g., PODate, InvoiceDate, PayDate) require conversion to datetime format for accurate time-based analysis and reporting.  

6. Common fields such as VendorNumber, VendorName, and Brand can serve as reliable join keys. Recommendation: Standardize these fields across datasets to prevent mismatches during merges.  

7. Initial insights:  
* PurchasePrice and ActualPrice show variability across vendors and brands, suggesting potential for price optimization strategies.  
* GrossProfit and ProfitMargin in `vendor_sales_summary` highlight high-performing vendors and areas where cost management could improve margins.  
* TotalSalesQuantity versus TotalPurchaseQuantity can reveal stock turnover efficiency and help in inventory planning.  

8. Recommendations for further processing:  
* Impute or correct missing values, especially in `vendor_invoice.Approval`, to ensure complete invoice validation.  
* Standardize date formats and convert to datetime objects for accurate temporal analysis.  
* Normalize categorical fields like VendorName and Description to maintain consistency across datasets.  
* Conduct outlier detection on numeric fields such as PurchasePrice, Dollars, and Quantity to identify anomalies or data entry errors.  
* Consider merging datasets to create a consolidated view for comprehensive analysis of vendor performance, pricing trends, and procurement efficiency.  
* Perform exploratory data analysis (EDA) including correlation and trend analysis to uncover actionable insights for procurement optimization.






